<a href="https://colab.research.google.com/github/Rahi2208/Transformers-Causal-Decoder-nanoGPT-from-scratch-/blob/main/rahi_Transformers_causal_decoder_only_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-05 01:44:26--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-05 01:44:26 (33.1 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
import torch
from torch.nn import functional as F
import numpy as np
import torch.nn as nn

In [13]:
#input data
with open('input.txt') as f:
  text = f.read()

#device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
if device == 'cuda':
  torch.cuda.manual_seed(1337)

#set parameter
block_size= 64
batch_size = 128
n_embd = 256
n_layer = 16
n_head = 16
test_iters = 8000
test_interval = 200
dropout = 0.0

#B = batch_size, T= block_size, C=n_embd
#set vocab
vocab = sorted(list(set(text)))
vocab_size = len(vocab)

#built encoder and decoder
stoi = {ch:i for i,ch in enumerate(vocab)}
itos = {i:ch for i,ch in enumerate(vocab)}
encoder = lambda s: [stoi[text] for text in s]
decoder = lambda s: [itos[idx] for idx in s]
data = encoder(text)

#split data (data must be tensor)
holdout = int(0.9*len(data))
train_data = torch.tensor(data[:holdout])
test_data = torch.tensor(data[holdout:])

#get_batch for SGD
def get_batch(split):
  data_used = train_data if split == 'train' else test_data
  ix = torch.randint(len(data_used)-block_size, (batch_size,)) #B
  x = torch.stack([data_used[i:i+block_size] for i in ix]) #B T
  y = torch.stack([data_used[i+1: i+1+block_size] for i in ix]) #B T
  x = x.to(device)
  y = y.to(device)
  return x,y

#estimate_loss
@torch.no_grad()
def estimate_loss_and_accuracies():
  model.eval()
  out={}
  for split in ['train', 'test']:
    losses = torch.zeros(test_iters)
    acc = torch.zeros(test_iters)
    for k in range(test_iters):
      x,y = get_batch(split)

      #estimate loss
      logits, loss = m(x,y) #if returning value from module, use .item() to get the value
      cost = loss.item()
      losses[k] = cost
      B,T = y.shape

      #estimate accuracy (correct/total) *100
      preds = torch.argmax(logits, dim=-1) #find absolute answer, so y=the largest probability
      y = y.reshape(B*T)
      correct = (preds == y).sum().item() #preds = (B*T), y = (B,T), we need to reshape y first so the shape is the same
      total = y.numel()
      accuracies = (correct/total) *100
      acc[k] = accuracies
    out[split] = [losses.mean(), acc.mean()]
  return out

#TCDE architecture

#self attention (softmax((Q.KT)/))
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    #matrix multiplication + modify the head
    self.key = nn.Linear(n_embd, head_size, bias=False) #we change the layer size to maximize the learning process
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #set buffer torch.tril with maximum size of block_size
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    K = self.key(x)
    Q = self.query(x)

    wei = Q@K.transpose(-2,-1)*head_size**-0.5 #K @ Q.transpose / √(C), shape: B T C @ B C T = B T T ([[[T], . . . ], [[T], . . ]]), thats why the buffer's sie is [T,T]
    wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf'))  #we still need to slice self.tril because buffer is not always block_size x block_size, its just the max sise, so we need to slice it until the input shape T
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    V = self.value(x)
    out = wei @ V
    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.mulhead = nn.ModuleList([Head(head_size) for _ in range(n_head)]) #as a vast to register multi modules into one = nn.ModuleList()
    self.proj = nn.Linear(n_embd, n_embd) #projection is not changing the layer size
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.mulhead], dim=-1) #B T head_size*4, dimension is static.
    return self.dropout(self.proj(out))

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd),
        nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.network(x)

class TransformersBlock(nn.Module):
  def __init__(self):
    super().__init__()
    head_size = n_embd//n_head
    self.mulhead = MultiHeadAttention(head_size)
    self.net = FeedForward()
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    #transformersblock = connecting all transformers architecture using residual network
    x = x+self.mulhead(self.ln1(x))
    out = x+self.net(self.ln2(x)) #B T C
    return out

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.Block = TransformersBlock()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.pos_embedding_table = nn.Embedding(block_size, n_embd)
    self.ln_head = nn.LayerNorm(n_embd) #final layer normalization
    self.output_layer = nn.Linear(n_embd, vocab_size)

  def forward(self, input_batch, Targets=None):
    B,T = input_batch.shape
    token_emb = self.token_embedding_table(input_batch) #B T C, [[[C], . . ], [[C], . . ], . . . ,]
    pos_emb = self.pos_embedding_table(torch.arange(T, device=device)) #T C, [[C], . . . ]
    #we need to use torch.arange as an input data for positional embedding table because we want to assign the index in order
    x = token_emb + pos_emb
    x = self.Block(x)
    x = self.ln_head(x)
    logits = self.output_layer(x) #B T vocab_size, logits = the numbers in the output layer

    #calculating the logits and loss
    if Targets==None:
      loss=None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = Targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
      return logits, loss

  def generate(self, idx, max_new_tokens):
    idx_cond = idx[:, -block_size:] #take the last 32 tokens
    logits, loss = self(idx_cond) #procees with idx_cond
    logits = logits[:, -1, :] #set until the last word
    pred = F.softmax(logits, dim=-1) #predict the next word
    idx_next = torch.multinomial(logits, num_samples=1) #find the most suitable answer based on probabilities(vary)
    return torch.cat((idx, idx_next), dim=1) #concatenate the answer in horizontal way so it looks like sentence

#train the model
model = BigramLanguageModel()
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = 0.001)

for iters in range(test_iters):
  if iters % test_interval == 0 or test_iters-1 == iters:
    cost = estimate_loss_and_accuracies()
    print(f'epochs: {iters+1}, train_acc = {cost['train'][1]:.4f},  train_loss: {cost['train'][0]:.4f}, test_acc = {cost['test'][1]:.4f}, test_loss = {cost['test'][0]:.4f}')

  x,y = get_batch('train')
  optimizer.zero_grad()
  logits, loss = m(x,y)
  loss.backward()
  optimizer.step()

#generate text
context = torch.ones((1,1), dtype = torch.long, device=device)
print(decoder(m.generate(context, max_new_tokens=2000)))


epochs: 1, train_acc = 3.0164,  train_loss: 4.3088, test_acc = 2.9775, test_loss = 4.3142
epochs: 201, train_acc = 28.6637,  train_loss: 2.5087, test_acc = 28.4684, test_loss = 2.5166
epochs: 401, train_acc = 32.3945,  train_loss: 2.3435, test_acc = 32.1963, test_loss = 2.3474
epochs: 601, train_acc = 34.1562,  train_loss: 2.2635, test_acc = 33.8178, test_loss = 2.2705
epochs: 801, train_acc = 36.1188,  train_loss: 2.2008, test_acc = 35.1400, test_loss = 2.2215
epochs: 1001, train_acc = 37.0279,  train_loss: 2.1519, test_acc = 36.0094, test_loss = 2.1753
epochs: 1201, train_acc = 38.0381,  train_loss: 2.1088, test_acc = 36.4657, test_loss = 2.1533
epochs: 1401, train_acc = 38.4994,  train_loss: 2.0840, test_acc = 36.8894, test_loss = 2.1315
epochs: 1601, train_acc = 39.5430,  train_loss: 2.0537, test_acc = 37.7734, test_loss = 2.1125
epochs: 1801, train_acc = 39.7738,  train_loss: 2.0358, test_acc = 37.9210, test_loss = 2.0962
epochs: 2001, train_acc = 40.6339,  train_loss: 2.0066, tes

KeyboardInterrupt: 